# 01 – EC3D Data Exploration
	•	Obiettivi:
	•	- Caricare il dataset EC3D
	•	- Capire la struttura dei dati (forme, chiavi, label)
	•	- Contare le istanze per classe ed esercizio
	•	- Definire uno split cross-subject (train/test)
	•	- Creare il mapping label → frase breve (per il contrastivo)

data_3D.pickle → è il file usato nel paper, contiene le label e le coordinate 3D di tutte le sequenze, con shape (29789, 3, 25) (frame, xyz, 25 joint). È quello da usare per riprodurre gli esperimenti.

data.pickle → contiene tutti i dati grezzi (parametri di camera, 2D e 3D, ecc.).

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

# Cartella root del progetto (un livello sopra "notebooks/")
ROOT_DIR = Path("..").resolve()

# Path del file EC3D
DATA_PATH = ROOT_DIR / "data" / "EC3D" / "data_3D.pickle"
print("DATA_PATH:", DATA_PATH)
print("Esiste?", DATA_PATH.exists())

In [ ]:
with open(DATA_PATH, "rb") as f:
    data = pickle.load(f)

print("Tipo di data:", type(data))

In [ ]:
print("Chiavi del dizionario:", list(data.keys()))
first_key = list(data.keys())[0]
second_key = list(data.keys())[1] 
print("Tipo del primo elemento:", type(data[first_key])) 
print("Tipo del secondo elemento:", type(data[second_key]))

	•	poses shape: (29789, 3, 25)
	•	29789 = numero totale di frame di tutti gli esercizi messi insieme
	•	3 = coordinate (x, y, z)
	•	25 = joint dello scheletro
Quindi ogni poses[i] è una matrice 3×25:
riga = coordinata (x/y/z), colonna = joint.
	•	labels shape: (29789, 5) dtype stringa
	•	ogni riga di labels[i] contiene 5 campi testuali che descrivono quel frame.
	•	dai primi 10:

	•	colonna 0 → tipo esercizio (SQUAT, poi vedremo anche LUNGE, PLANK)
	•	colonna 1 → soggetto (Hugues, Sena, Vidit)
	•	le colonne 2–4 sono ID numerici (stringhe), che codificano combinazioni di:
	•	variante / tipo di movimento,
	•	ID della sequenza / prova,
	•	indice del frame.
Lo capiamo guardando le uniche per colonna.


In [ ]:
poses = data["poses"]
labels = data["labels"]

print("poses shape:", poses.shape)
print("labels shape:", labels.shape)
print("poses dtype:", poses.dtype)
print("labels dtype:", labels.dtype)

In [ ]:
print("Prime 10 label:", labels[:10])

unique_labels, counts = np.unique(labels, return_counts=True)
print("Label uniche:", unique_labels)
print("Frequenze corrispondenti:", counts)

In [ ]:
first_pose = poses[0]
print("Shape prima pose:", first_pose.shape)
print("Valori di esempio (prime 2 joint):")
print(first_pose[:, :2])  

Creiamo un DataFrame con colonne generiche e guardiamo le prime righe

	•	ogni riga di labels_df descrive un frame;
	•	una sequenza (una rip o una clip intera) è data da tutti i frame che condividono:
	•	stesso exercise
	•	stesso subject
	•	stessa c2 (tipo di esecuzione: corretta o errore X)
	•	stessa c3 (ID prova).

c4 è solo un indice di frame (per ordinarli correttamente nel tempo).

In [ ]:
labels_df = pd.DataFrame(
    labels,
    columns=["exercise", "subject", "c2", "c3", "c4"]
)

labels_df.head(10)

Guardiamo i valori unici per colonna (così esploriamo più nel profondo anche c2, c3, c4)

In [ ]:
for col in labels_df.columns:
    print(f"\nColonna: {col}")
    print("  # valori unici:", labels_df[col].nunique())
    print("  Alcuni valori:", labels_df[col].unique())

Contiamo le combinazioni più frequenti (per intuire la struttura “sequenza”)

In [ ]:
print("Combinazioni exercise/subject/c2 più frequenti:")
print(
    labels_df
    .groupby(["exercise", "subject", "c2"])
    .size()
    .sort_values(ascending=False)
    
)

In [ ]:
# Ogni riga: una combinazione (exercise, subject, c2, c3)
# e il numero di frame in quella sequenza (non ci interessa quanto, solo che esista)
seq_counts = (
    labels_df
    .groupby(["exercise", "subject", "c2", "c3"])["c4"]
    .nunique()
    .reset_index(name="n_frames")
)

# Adesso contiamo quante sequenze (c3 diversi) ci sono per (exercise, subject, c2)
instr_seq_counts = (
    seq_counts
    .groupby(["exercise", "subject", "c2"])["c3"]
    .nunique()
    .reset_index(name="n_sequences")
)

instr_seq_counts.sort_values(["exercise", "subject", "c2"], inplace=True)
instr_seq_counts.head(30)

In [ ]:
squats = instr_seq_counts[instr_seq_counts["exercise"] == "SQUAT"]
squats

In [ ]:
# 1) Converto le colonne numeriche in interi
labels_df["instruction_id"] = labels_df["c2"].astype(int)
labels_df["trial_id"] = labels_df["c3"].astype(int)
labels_df["frame_idx"] = labels_df["c4"].astype(int)

# 2) Mapping instruction_id -> nome errore, preso dal codice ufficiale
INSTRUCTION_ID_TO_NAME = {
    1: 'Correct',
    2: 'Feets too wide',
    3: 'Knees inward',
    4: 'Not low enough',
    5: 'Front bended',
    6: 'Knees pass toes',
    7: 'Banana back',
    8: 'Rolled back',
    9: 'Asymmetric',
    10: 'Unknown',
}

labels_df["instruction_name"] = labels_df["instruction_id"].map(INSTRUCTION_ID_TO_NAME)

#printa labels_df dove instruction name è asymmetric
print(labels_df[labels_df["instruction_name"] == "Asymmetric"])
#non abbiamo alcun esempio di asymmetric nel dataset

labels_df.head(100)  


In [ ]:
# 3) Mapping (exercise, instruction_id) -> global class id (0..11)
GLOBAL_LABEL_MAP = {
    ('SQUAT', 1): 0,   # Correct
    ('SQUAT', 2): 1,   # Feets too wide
    ('SQUAT', 3): 2,   # Knees inward
    ('SQUAT', 4): 3,   # Not low enough
    ('SQUAT', 5): 4,   # Front bended
    ('SQUAT', 10): 5,  # Unknown

    ('Lunges', 1): 6,  # Correct
    ('Lunges', 4): 7,  # Not low enough
    ('Lunges', 6): 8,  # Knees pass toes

    ('Plank', 1): 9,   # Correct
    ('Plank', 7): 10,  # Banana back
    ('Plank', 8): 11,  # Rolled back
}

def map_global_label(row):
    key = (row["exercise"], int(row["instruction_id"]))
    return GLOBAL_LABEL_MAP.get(key, -1)  # -1 se per qualche motivo non mappa

labels_df["global_label_id"] = labels_df.apply(map_global_label, axis=1)

# Nome leggibile tipo "SQUAT - Knees inward"
labels_df["global_label_name"] = (
    labels_df["exercise"] + " - " + labels_df["instruction_name"]
)

labels_df.tail(60)

Verifico che la tabella del paper sia coerente con il dataset

In [ ]:
# Ogni sequenza è identificata da (exercise, subject, instruction_id, c3)
labels_df["sequence_key"] = (
    labels_df["exercise"] + "_" +
    labels_df["subject"] + "_" +
    labels_df["instruction_id"].astype(str) + "_" +
    labels_df["c3"]
)

# Per ogni sequenza, contiamo i frame (non fondamentale, ma utile)
seq_frame_counts = (
    labels_df
    .groupby(["exercise", "subject", "instruction_id", "instruction_name", "c3"])["c4"]
    .nunique()
    .reset_index(name="n_frames")
)

# Ora per il confronto col paper ci basta quante sequenze per (exercise, subject, instruction)
seq_counts = (
    seq_frame_counts
    .groupby(["exercise", "subject", "instruction_id", "instruction_name"])["c3"]
    .nunique()
    .reset_index(name="n_sequences")
)

seq_counts.sort_values(["exercise", "subject", "instruction_id"], inplace=True)
seq_counts

Il mapping originale definisce anche 9 → Asymmetric,

ma nel file reale data_3D.pickle non compare mai, quindi la classe “Asymmetric” è teorica ma non presente nel dataset rilasciato.


In [ ]:
instr_ids = labels_df["c2"].astype(int).values
unique_ids, counts = np.unique(instr_ids, return_counts=True)
print("Instruction_id unici:", unique_ids)
print("Frequenze:", counts)
print("Presenza id=9:", (instr_ids == 9).sum())

## Missione 3 – Costruire le sequenze

Ogni sequenza è identificata da: `(exercise, subject, instruction_id, trial_id)`

Raggruppiamo i frame per sequenza ed estraiamo gli array di pose corrispondenti.


In [ ]:
# Chiave unica per ogni sequenza
labels_df["sequence_key"] = (
    labels_df["exercise"] + "_" +
    labels_df["subject"] + "_" +
    labels_df["instruction_id"].astype(str) + "_" +
    labels_df["trial_id"].astype(str)
)

group_cols = [
    "exercise",
    "subject",
    "instruction_id",
    "instruction_name",
    "global_label_id",
    "trial_id",
    "sequence_key",
]

sequence_groups = labels_df.groupby(group_cols)

sequences = []  # lista di np.array (T, 3, 25)
targets = []   # lista di global_label_id (int)
meta = []      # info per ogni sequenza

for (exercise, subject, instr_id, instr_name, glob_id, trial_id, seq_key), group in sequence_groups:
    group_sorted = group.sort_values("frame_idx")
    indices = group_sorted.index.to_numpy()
    
    # estraggo le pose dei frame di quella sequenza
    seq_poses = poses[indices]  # shape (T, 3, 25)
    
    sequences.append(seq_poses.astype(np.float32))
    targets.append(int(glob_id))
    meta.append({
        "sequence_key": seq_key,
        "exercise": exercise,
        "subject": subject,
        "instruction_id": int(instr_id),
        "instruction_name": instr_name,
        "global_label_id": int(glob_id),
        "global_label_name": f"{exercise} - {instr_name}",
        "trial_id": int(trial_id),
        "num_frames": int(len(group_sorted)),
    })

print("Numero di sequenze totali:", len(sequences))
print("Esempio meta:", meta[0])


## Missione 4 – Salvare i file processati

### 4a) Salvare `ec3d_sequences.pkl`

Questo file contiene:
- `sequences`: lista di array numpy (T, 3, 25)
- `labels`: array di global_label_id (0-11)
- `meta`: lista di dizionari con info per ogni sequenza


In [ ]:
import json

DATA_DIR = ROOT_DIR / "data" / "EC3D"

ec3d_data = {
    "sequences": sequences,                           # lista di np.array (T, 3, 25)
    "labels": np.array(targets, dtype=np.int64),      # global_label_id
    "meta": meta,
}

OUT_PATH = DATA_DIR / "ec3d_sequences.pkl"
with open(OUT_PATH, "wb") as f:
    pickle.dump(ec3d_data, f)

print("✅ Salvato:", OUT_PATH)
print(f"   - {len(sequences)} sequenze")
print(f"   - {len(np.unique(targets))} classi uniche")


### 4b) Creare lo split cross-subject

Split train/test basato sui soggetti:
- **Train**: Hugues, Sena, Isinsu
- **Test**: Vidit


In [ ]:
TRAIN_SUBJECTS = ["Hugues", "Sena", "Isinsu"]
TEST_SUBJECTS = ["Vidit"]

train_indices = [i for i, m in enumerate(meta) if m["subject"] in TRAIN_SUBJECTS]
test_indices = [i for i, m in enumerate(meta) if m["subject"] in TEST_SUBJECTS]

print("Numero sequenze train:", len(train_indices))
print("Numero sequenze test:", len(test_indices))

# Calcolo distribuzione per classe
train_labels = [targets[i] for i in train_indices]
test_labels = [targets[i] for i in test_indices]

print("\nDistribuzione classi train:")
for label_id in sorted(set(train_labels)):
    count = train_labels.count(label_id)
    label_name = meta[train_indices[train_labels.index(label_id)]]["global_label_name"]
    print(f"  Classe {label_id} ({label_name}): {count} sequenze")

print("\nDistribuzione classi test:")
for label_id in sorted(set(test_labels)):
    count = test_labels.count(label_id)
    label_name = meta[test_indices[test_labels.index(label_id)]]["global_label_name"]
    print(f"  Classe {label_id} ({label_name}): {count} sequenze")

split = {
    "train_subjects": TRAIN_SUBJECTS,
    "test_subjects": TEST_SUBJECTS,
    "train_indices": train_indices,
    "test_indices": test_indices,
}

SPLIT_PATH = DATA_DIR / "split_cross_subject.json"
with open(SPLIT_PATH, "w") as f:
    json.dump(split, f, indent=2)

print(f"\n✅ Salvato split: {SPLIT_PATH}")


## ✅ Riepilogo
Completato l'esplorazione e il preprocessing del dataset EC3D:

1. **Caricato** `data_3D.pickle` (29789 frame, 25 joint)
2. **Creato** `labels_df` con colonne interpretabili:
   - `instruction_id`, `trial_id`, `frame_idx`
   - `instruction_name` (es. "Correct", "Knees inward")
   - `global_label_id` (0-11) e `global_label_name`
3. **Costruito** le sequenze: raggruppato i frame per ottenere array (T, 3, 25)
4. **Salvato**:
   - `ec3d_sequences.pkl` → contiene sequences, labels, meta
   - `split_cross_subject.json` → train/test split per soggetti

**Prossimi passi**: nei notebook successivi useremo questi file per:
- Addestrare modelli di classificazione
- Implementare contrastive learning con CLIP
- Generare feedback testuale
